## Load latest headlines

In [1]:
# ========== Bootstrap: ensure required Python packages are present =========
import importlib, subprocess, sys
from typing import Optional

def _ensure(pkg_name: str, import_name: Optional[str] = None):
    try:
        importlib.import_module(import_name or pkg_name)
    except ModuleNotFoundError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg_name])
    finally:
        globals()[import_name or pkg_name] = importlib.import_module(import_name or pkg_name)

_ensure('pandas')
print('All dependencies ready.\n')


All dependencies ready.



In [2]:
from pathlib import Path
archive = Path('archive')
archive.mkdir(exist_ok=True)
import pandas as pd
latest = pd.read_csv('../headlines/latest.csv')
latest.head()

,pubdate,source,title,link
0,2026-11-11-00-55-35 +0000,nypost,A 300-year-old Torah scroll that survived the ...,https://nypost.com/2026/11/10/us-news/a-300-ye...
1,2026-11-11-00-48-37 +0000,nyt,Trump Officials Drop Threat to Colorado School...,https://www.nytimes.com/2026/11/10/us/politics...
2,2026-11-11-00-47-53 +0000,nypost,"SPLC investigation started under Biden, prompt...",https://nypost.com/2026/11/10/us-news/splc-inv...
3,2026-11-11-00-46-46 +0000,nyt,Ballot Deadlines Pass in Mississippi Senate Ra...,https://www.nytimes.com/2026/11/10/us/election...
4,2026-11-11-00-43-15 +0000,nyt,Republicans Turn Back Effort to Impeach Hegseth,https://www.nytimes.com/2026/11/10/us/politics...


## Calculate word frequencies

In [3]:
import re
from collections import Counter
from datetime import datetime

with open('exclude.txt') as f:
    stop_words = set(w.strip() for w in f if w.strip())
words = re.findall(r'[A-Za-z]+', ' '.join(latest['title']).lower())
filtered = [w for w in words if w not in stop_words and len(w) > 1]
counts = Counter(filtered)
score_df = (
    pd.DataFrame(counts.items(), columns=['word','score'])
    .sort_values('score', ascending=False)
)
score_df[['score','word']].to_csv('scores.csv', index=False)
timestamp = datetime.utcnow().strftime('%Y-%m-%d-%H-00-00')
score_df[['score','word']].to_csv(archive / f'scores-{timestamp}.csv', index=False)
score_df.head()


/tmp/ipykernel_2661/2220774760.py:15: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  timestamp = datetime.utcnow().strftime('%Y-%m-%d-%H-00-00')


,word,score
8,trump,32
258,nvidia,29
256,huang,24
59,house,21
6,new,21


## Rank headlines by score

In [4]:
word_scores = dict(score_df[['word','score']].values)
latest['score'] = latest['title'].apply(
    lambda t: sum(
        word_scores.get(w.lower(), 0)
        for w in re.findall(r'[A-Za-z]+', t)
        if len(w) > 1
    )
)
ranked = latest.sort_values('score', ascending=False)
ranked[['score','pubdate','source','title','link']].to_csv('rank.csv', index=False)
ranked[['score','pubdate','source','title','link']].to_csv(archive / f'rank-{timestamp}.csv', index=False)
ranked.head()


,pubdate,source,title,link,score
30,2026-11-11-00-07-28 +0000,nypost,Chinese ‘spy’ Alice Huang met Nvidia CEO Jense...,https://nypost.com/2026/11/10/business/alice-h...,168
357,2026-11-10-11-59-50 +0000,nypost,FBI is ‘looking into’ Nvidia boss Jensen Huang...,https://nypost.com/2026/11/10/us-news/fbi-is-l...,162
199,2026-11-10-19-21-16 +0000,nypost,Nvidia loses nearly $450B in two-day wipeout a...,https://nypost.com/2026/11/10/business/nvidia-...,154
169,2026-11-10-20-22-14 +0000,nypost,Jensen Huang’s ties to suspected Chinese spy h...,https://nypost.com/2026/11/10/business/jensen-...,142
127,2026-11-10-21-35-09 +0000,nypost,Nvidia CEO Jensen Huang’s ‘spy’ affair could s...,https://nypost.com/2026/11/10/business/nvidia-...,140


## Select top headlines

In [5]:
with open('exclude_sources.txt') as f:
    excluded_sources = {
        line.strip()
        for line in f
        if line.strip() and not line.strip().startswith('#')
    }

top_count = 10
top_rows = []
working = word_scores.copy()
remaining = latest.copy()

while len(top_rows) < top_count and not remaining.empty:
    scored_remaining = remaining.assign(
        score=remaining['title'].apply(
            lambda t: sum(
                working.get(w.lower(), 0)
                for w in re.findall(r'[A-Za-z]+', t)
                if len(w) > 1
            )
        )
    )
    ranked_loop = scored_remaining.sort_values('score', ascending=False)

    if ranked_loop.empty:
        break

    top_story = ranked_loop.iloc[0]
    remaining = remaining.drop(top_story.name)

    if top_story['source'] in excluded_sources:
        continue

    words = set(re.findall(r'[A-Za-z]+', top_story['title'].lower()))
    for word in words:
        working.pop(word, None)

    top_rows.append(top_story[['score', 'pubdate', 'source', 'title', 'link']])

top_df = pd.DataFrame(top_rows)
top_df.to_csv('top.csv', index=False)
top_df.to_csv(archive / f'top-{timestamp}.csv', index=False)
top_df


,score,pubdate,source,title,link
30,168,2026-11-11-00-07-28 +0000,nypost,Chinese ‘spy’ Alice Huang met Nvidia CEO Jense...,https://nypost.com/2026/11/10/business/alice-h...
48,130,2026-11-10-23-31-56 +0000,nypost,Trump White House unites behind Pete Hegseth a...,https://nypost.com/2026/11/10/us-news/white-ho...
73,92,2026-11-10-22-59-00 +0000,wsj,House Republicans elected Rep. Jim Jordan of O...,https://www.wsj.com/politics/jim-jordan-wins-s...
446,85,2026-11-10-03-28-00 +0000,wsj,California voters rejected a proposed billiona...,https://www.wsj.com/politics/elections/califor...
14,79,2026-11-11-00-29-45 +0000,latimes,Meghan Markle's father Thomas Markle Sr. cut o...,https://www.latimes.com/entertainment-arts/sto...
389,63,2026-11-10-10-48-00 +0000,nyt,Here’s why last year’s government shutdown wil...,https://www.nytimes.com/live/2026/11/10/busine...
250,63,2026-11-10-17-36-51 +0000,nypost,Gas prices drop below $4 a gallon for first ti...,https://nypost.com/2026/11/10/business/gas-pri...
316,60,2026-11-10-14-20-09 +0000,cbc,Anthropic eyes $1 trillion US valuation in hot...,https://www.cbc.ca/news/business/anthropic-ipo...
265,52,2026-11-10-17-00-11 +0000,bbc,Inside devastated front-line city that Russia ...,https://www.bbc.co.uk/news/articles/cq0memrywv...
457,51,2026-11-10-01-53-00 +0000,wsj,The House of Representatives is barreling towa...,https://www.wsj.com/politics/national-security...


## Create JSON version of top headlines

In [6]:
import pandas as pd
pd.read_csv('top.csv').to_json('top.json', orient='records', indent=2)
